In [6]:
import os
import warnings
import logging
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

# Suppress warnings and TF logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

import sys
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception as e:
    pass

from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline
import data_utils

try:
    from deepforest import CascadeForestClassifier
    DEEPFOREST_AVAILABLE = True
except ImportError:
    DEEPFOREST_AVAILABLE = False
    print("Warning: deepforest not installed. Run `pip install deep-forest` to use Deep Forest models.")

# Import the custom extractor (ensure v1_vibration_extractor.py is in the same directory)
from v1_vibration_extractor import SensorFeatureExtractor

In [ ]:
# ============================================================
# CONFIGURATION — switch feature + classifier pipelines here
# ============================================================
TRAIN_RF = True
TRAIN_ET = True          # Extra Trees ("extra")
TRAIN_BOOST = True       # Histogram Gradient Boosting ("boost")
TRAIN_DF = True          # Deep Forest

# Feature extraction mode
# 'spectral_only': Applies FFT band features to IMU, stats to ToF/Thermo
# 'full_spectral': Applies FFT band features to ALL sensors (creative approach for ToF/Thermo)
FEATURE_MODE = "spectral_only"

target_col = "bfrb"
search_mode = "grid"  # "grid" or "bayesian"
random_state = 42
n_splits = 3
train_size = 0.2  # lower for quick local tests; raise for full runs
error_score_constant = 0.0
verbose = 2
do_cross_val = False

# Use sample for fast smoke tests (set False for full dataset)
use_sample = True
sample_pct = 0.05

if search_mode == "bayesian":
    try:
        from skopt import BayesSearchCV
        from skopt.space import Categorical, Integer, Real
        SKOPT_AVAILABLE = True
    except ImportError:
        SKOPT_AVAILABLE = False
        search_mode = "grid"
        print("Fallback to Grid Search: skopt not available.")

if do_cross_val:
    cv_object = GroupKFold(n_splits=n_splits)
else:
    cv_object = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

In [8]:
# ============================================================
# DATA LOADING & PREPROCESSING
# ============================================================
data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")

train_df = raw_train_df.set_index("row_id").copy(deep=True)

# Handedness correction
train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
left_handed_mask = train_df["handedness"].eq(0)
train_df.loc[left_handed_mask, "acc_x"] *= -1.0

# Upside-down correction
upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0
train_df = train_df.drop(columns=["handedness"])

# Targets
train_df["is_target"] = (train_df["sequence_type"] == "Target").astype(bool)
train_df["bfrb"] = train_df["gesture"].where(train_df["is_target"], "non_bfrb")

# Split
train_sample_df, hold_out_df = data_utils.sample_balanced_split(
    train_df, train_pct=train_size, test_pct=min(0.18, 1 - train_size), random_state=random_state
)

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
y_train = train_sample_df[["sequence_id", "is_target", "bfrb"]].copy()
y_test = hold_out_df[["sequence_id", "is_target", "bfrb"]].copy()
groups = X_train["sequence_id"]

Using local data folder: C:\Users\maran\OneDrive\Documents\Git Profile\cmi_dexter\data
Train: 1458 seqs | 17.9%
Test:  1452 seqs  | 17.8%


In [9]:
# ============================================================
# PARAMETER SPACE DEFINITION — INCLUDES FEATURE EXTRACTION
# ============================================================

# Base extractor settings (used as defaults if not overridden by search)
extractor_kwargs = {
    "sampling_rate": 50.0,
    "max_no_peaks": 8,
    "peaks_prominence": 1e-3,
    "apply_spectral_to_low_freq": (FEATURE_MODE == "full_spectral")
}

if search_mode == "bayesian" and SKOPT_AVAILABLE:
    # Shared feature extractor search space for Bayesian Optimization
    extractor_bayes_space = {
        "extractor__sampling_rate": Real(20.0, 300.0, prior="uniform"),
        "extractor__max_no_peaks": Integer(3, 20),
        "extractor__peaks_prominence": Real(1e-5, 1e-2, prior="log-uniform"),
    }

    # 1. Random Forest
    rf_param_space = {
        **extractor_bayes_space,
        "classifier__n_estimators": Integer(50, 4000),
        "classifier__max_depth": Integer(5, 400),
        "classifier__min_samples_split": Integer(2, 300),
        "classifier__min_samples_leaf": Integer(1, 200),
        "classifier__max_features": Categorical(["sqrt", "log2", None]),
        "classifier__class_weight": Categorical(["balanced", "balanced_subsample", None]),
    }

    # 3. Histogram Gradient Boosting
    boost_param_space = {
        **extractor_bayes_space,
        "classifier__max_iter": Integer(50, 2000),
        "classifier__learning_rate": Real(0.01, 0.3, prior="log-uniform"),
        "classifier__max_depth": Integer(3, 500),
        "classifier__l2_regularization": Real(0.0, 20.0, prior="log-uniform"),
        "classifier__min_samples_leaf": Integer(10, 500),
    }

    # 4. Deep Forest (Cascade Forest)
    df_param_space = {
        **extractor_bayes_space,
        "classifier__n_estimators": Integer(1, 2000),       # Number of cascades
        "classifier__max_depth": Integer(3, 300),          # Max depth of trees in forest
        "classifier__n_trees": Integer(50, 5000),          # Number of trees per layer
        "classifier__criterion": Categorical(["gini", "entropy"]),
    }

else:  # GRID SEARCH
    # Shared feature extractor grid space
    extractor_grid_space = {
        "extractor__sampling_rate": [20.0],
        "extractor__max_no_peaks": [15],
        "extractor__peaks_prominence": [1e-4],
    }

    # 1. Random Forest
    rf_param_space = {
        **extractor_grid_space,
        "classifier__n_estimators": [1000],
        "classifier__max_depth": [200],
        "classifier__min_samples_split": [50],
        "classifier__min_samples_leaf": [50],
        "classifier__max_features": ["sqrt"],
        "classifier__class_weight": ["balanced"],
    }

    # 3. Histogram Gradient Boosting
    boost_param_space = {
        **extractor_grid_space,
        "classifier__max_iter": [100],
        "classifier__learning_rate": [ 0.1],
        "classifier__max_depth": [20],
        "classifier__l2_regularization": [1.0],
        "classifier__min_samples_leaf": [100],
    }

    # 4. Deep Forest (Cascade Forest)
    df_param_space = {
        **extractor_grid_space,
        "classifier__n_estimators": [6],
        "classifier__max_depth": [10],
        "classifier__n_trees": [200],
        "classifier__criterion": ["entropy"],
    }

In [14]:
# ============================================================
# TRAINING & EVALUATION LOOP
# ============================================================
results_list = []
fitted_models = {}

def make_pipeline(classifier, param_space, name):
    extractor = SensorFeatureExtractor(**extractor_kwargs)
    pipe = Pipeline(steps=[
        ("extractor", extractor),
        ("classifier", classifier)
    ])
    
    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        search = BayesSearchCV(
            pipe, param_space, n_iter=10, scoring="f1",
            cv=GroupKFold(n_splits=n_splits), n_jobs=-1, 
            random_state=random_state, error_score=error_score_constant, verbose=verbose
        )
    else:
        search = GridSearchCV(
            pipe, param_space, scoring="f1",
            cv=GroupKFold(n_splits=n_splits), n_jobs=-1, 
            error_score=error_score_constant, verbose=verbose
        )
    return search

if TRAIN_RF:
    print("\n--- Training Baseline: Random Forest ---")
    rf_pipe = make_pipeline(RandomForestClassifier(random_state=random_state), rf_param_space, "RF")
    rf_pipe.fit(X_train, y_train[target_col], groups=groups)
    fitted_models["rf"] = rf_pipe.best_estimator_
    rf_preds = rf_pipe.predict(X_test)
    rf_f1 = f1_score(y_test[target_col], rf_preds)
    results_list.append({"Model": "Random Forest", "CV Score": rf_pipe.best_score_, "Holdout F1": rf_f1, "Best Params": rf_pipe.best_params_})
    print(f"RF CV: {rf_pipe.best_score_:.4f} | Holdout F1: {rf_f1:.4f}")

if TRAIN_BOOST:
    print("\n--- Training Baseline: Gradient Boosting ---")
    boost_pipe = make_pipeline(HistGradientBoostingClassifier(random_state=random_state), boost_param_space, "BOOST")
    boost_pipe.fit(X_train, y_train[target_col], groups=groups)
    fitted_models["boost"] = boost_pipe.best_estimator_
    boost_preds = boost_pipe.predict(X_test)
    boost_f1 = f1_score(y_test[target_col], boost_preds)
    results_list.append({"Model": "Hist Gradient Boost", "CV Score": boost_pipe.best_score_, "Holdout F1": boost_f1, "Best Params": boost_pipe.best_params_})
    print(f"BOOST CV: {boost_pipe.best_score_:.4f} | Holdout F1: {boost_f1:.4f}")

if TRAIN_DF and DEEPFOREST_AVAILABLE:
    print("\n--- Training Baseline: Deep Forest ---")
    df_pipe = make_pipeline(CascadeForestClassifier(random_state=random_state), df_param_space, "DF")
    df_pipe.fit(X_train, y_train[target_col], groups=groups)
    fitted_models["df"] = df_pipe.best_estimator_
    df_preds = df_pipe.predict(X_test)
    df_f1 = f1_score(y_test[target_col], df_preds)
    results_list.append({"Model": "Deep Forest", "CV Score": df_pipe.best_score_, "Holdout F1": df_f1, "Best Params": df_pipe.best_params_})
    print(f"DF CV: {df_pipe.best_score_:.4f} | Holdout F1: {df_f1:.4f}")
elif TRAIN_DF and not DEEPFOREST_AVAILABLE:
    print("Skipping Deep Forest — deep-forest package not installed.")


--- Training Baseline: Random Forest ---
Fitting 3 folds for each of 1 candidates, totalling 3 fits



KeyboardInterrupt



In [ ]:
# ============================================================
# FINAL SUMMARY + BEST MODEL HOLDOUT EVAL
# ============================================================
results_df = pd.DataFrame(results_list).sort_values("Holdout F1", ascending=False, na_position="last")
results_df.to_csv(results_dir / f"baselines_summary_{timestamp}.csv", index=False)

print("\n" + "=" * 60)
print("BASELINES SUMMARY")
print("=" * 60)
print(results_df[["Model", "CV Score", "Holdout F1"]].to_string(index=False))

# Full report for best holdout model
if len(results_df) > 0:
    best_name = results_df.iloc[0]["Model"].lower().replace(" ", "_").replace("hist_", "").replace("gradient_", "")
    key_map = {"random_forest": "rf", "extra_trees": "et", "boost": "boost", "deep_forest": "df"}
    model_key = key_map.get(best_name, list(fitted_models.keys())[0])
    
    if model_key in fitted_models:
        print(f"\nDetailed holdout eval for best model: {results_df.iloc[0]['Model']}")
        best_model = fitted_models[model_key]
        best_preds = best_model.predict(X_test)
        print("\nClassification Report:")
        print(classification_report(y_test["target"], best_preds, target_names=["Non-Target", "Target"]))